# CICCADA Data Calc Write Stage 2: Conformance Table Builders

Builds `conformance_voltvar_v2`, `conformance_voltwatt_v2`, `conformance_voltwattghi_v2`
from `ts` + `meta_up23c` + `all_uncurtailedpv_v2` (Stage 1 output).

**Run the cells in order.** Sections 1–4 are the smoke test on a single month /
single site-slice; do not skip to Section 5 until Section 4 comes back clean.

| Issue | Fixed in |
|---|---|
| R1 max(voltage) | `stage2_common.site_agg_cte` |
| R2 flex_export_detected = False | `stage2_common.meta_filter` (`exclude_flex=True`) |
| R3 / R9 AEST dates | `stage2_common.aest_month_window` + `temporal_cols` |
| R4 column naming | `build_conformance_voltvar` (thresholds unchanged) |
| R7 V-VAr curtailment zone | `build_conformance_voltvar` |
| R10 capability on S_99 | `as4777_curves.q_cap_absorbing_sql('P_kW','S_99')` |
| R11 curve keystone | both builders import, none re-implement |
| R12 AEST day/night | `stage2_common.temporal_cols` |
| R13 total_count | `build_conformance_voltwatt` (both tables agree) |
| R14 null_uncurtailed_P_count | both GHI-joined tables |
| R15 NULL not 0 | all curtailment columns |
| R16 2024 + 2025 | Section 5 |


## 0. Setup

In [8]:
import sys
import os
import time
import importlib
from pathlib import Path

# Point Python at shared/ and this Stage 2 folder.
ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "shared"))
sys.path.insert(0, str(ROOT / "data_calc_write" / "stage2_conformance"))

from aws_config import aq, tables, databases
from ciccada_config import SA, SAI
from stage2_common import aest_month_window

# Reload the curve keystone before reloading builders that import from it.
import as4777_curves
import build_conformance_voltvar as vv
import build_conformance_voltwatt as vw

as4777_curves = importlib.reload(as4777_curves)
vv = importlib.reload(vv)
vw = importlib.reload(vw)

DB = SAI
N_PARTS = 8

print(
    "targets:",
    "\n", vv.TARGET,
    "\n", vw.TARGET_BASIC,
    "\n", vw.TARGET_GHI,
)

print("\nVolt-VAr capability SQL:")
print(as4777_curves.q_cap_absorbing_sql("P_kW", "S_99"))

Using VVAR_V3 = 240.0 V (AS4777.2:2020 Australia A)
Using VW_V1 = 253.0 V (AS4777.2:2020 Australia A)
Using VW_V1 = 253.0 V (AS4777.2:2020 Australia A)
targets: 
 conformance_voltvar_v2 
 conformance_voltwatt_v2 
 conformance_voltwattghi_v2

Volt-VAr capability SQL:
CASE
            WHEN S_99 <= 0 THEN 0
            WHEN (power(S_99, 2) - power(abs(P_kW), 2)) <= 0 THEN 0
            ELSE -sqrt(power(S_99, 2) - power(abs(P_kW), 2))
        END


In [9]:
# Connection + Stage 1 dependency check.
# Iceberg tables return nothing from DESCRIBE -- use SELECT * LIMIT 1.
aq("SELECT count(*) AS n_rows, count(DISTINCT site_id) AS n_sites "
   "FROM all_uncurtailedpv_v2", database=DB)

,n_rows,n_sites
0,467332679,14299


In [10]:
stage1_keys = aq("""
    WITH key_counts AS (
        SELECT site_id, t_stamp, count(*) AS n
        FROM all_uncurtailedpv_v2
        GROUP BY site_id, t_stamp
    )
    SELECT
        count_if(n > 1) AS duplicated_keys,
        coalesce(sum(CASE WHEN n > 1 THEN n - 1 ELSE 0 END), 0) AS excess_rows,
        max(n) AS maximum_rows_per_key
    FROM key_counts
""", database=DB)

display(stage1_keys)

duplicated_keys = int(stage1_keys["duplicated_keys"].iloc[0])
excess_rows = int(stage1_keys["excess_rows"].iloc[0])
maximum_rows = int(stage1_keys["maximum_rows_per_key"].iloc[0])

assert duplicated_keys == 0, f"Stage 1 has {duplicated_keys:,} duplicated keys"
assert excess_rows == 0, f"Stage 1 has {excess_rows:,} excess rows"
assert maximum_rows <= 1, f"Maximum rows per key is {maximum_rows}"

print("Stage 1 uniqueness check passed.")

,duplicated_keys,excess_rows,maximum_rows_per_key
0,0,0,1


Stage 1 uniqueness check passed.


In [11]:
# Stage 2 metadata cardinality check.
#
# Stage 2 joins raw telemetry to meta_up23c by circuit_id. Each eligible
# circuit must resolve to exactly one site, polarity, capacity and S_99.
metadata_diagnostics = aq("""
    WITH eligible_metadata AS (
        SELECT DISTINCT
            circuit_id,
            site_id,
            circuit_polarity,
            ac_capacity_kw,
            s_99
        FROM meta_up23c
        WHERE is_pv = true
          AND flex_export_detected = false
          AND ac_capacity_kw > 0
          AND s_99 > 0
    ),
    per_circuit AS (
        SELECT
            circuit_id,
            count(*) AS n_variants,
            count(DISTINCT site_id) AS n_sites,
            count(DISTINCT circuit_polarity) AS n_polarities,
            count(DISTINCT ac_capacity_kw) AS n_capacities,
            count(DISTINCT s_99) AS n_s99_values,
            count_if(site_id IS NULL) AS null_site_rows,
            count_if(circuit_polarity IS NULL) AS null_polarity_rows
        FROM eligible_metadata
        GROUP BY circuit_id
    )
    SELECT
        count(*) AS eligible_circuits,
        count_if(n_variants > 1) AS circuits_with_variants,
        count_if(n_sites > 1) AS circuits_with_multiple_sites,
        count_if(n_polarities > 1) AS circuits_with_conflicting_polarity,
        count_if(n_capacities > 1) AS circuits_with_conflicting_capacity,
        count_if(n_s99_values > 1) AS circuits_with_conflicting_s99,
        count_if(null_site_rows > 0) AS circuits_with_null_site,
        count_if(null_polarity_rows > 0) AS circuits_with_null_polarity
    FROM per_circuit
""", database=DB)

display(metadata_diagnostics)

,eligible_circuits,circuits_with_variants,circuits_with_multiple_sites,circuits_with_conflicting_polarity,circuits_with_conflicting_capacity,circuits_with_conflicting_s99,circuits_with_null_site,circuits_with_null_polarity
0,26397,0,0,0,0,0,0,0


In [12]:
diag = metadata_diagnostics.iloc[0]

fatal_columns = [
    "circuits_with_multiple_sites",
    "circuits_with_conflicting_polarity",
    "circuits_with_conflicting_capacity",
    "circuits_with_conflicting_s99",
    "circuits_with_null_site",
    "circuits_with_null_polarity",
]

failures = {
    column: int(diag[column])
    for column in fatal_columns
    if int(diag[column]) != 0
}

assert not failures, f"Ambiguous Stage 2 circuit metadata: {failures}"

print("Stage 2 metadata cardinality check passed.")

Stage 2 metadata cardinality check passed.


## 1. Read the SQL before running it


`preview_sql` builds the exact INSERT for one slice without executing it.


Check the AEST window: for AEST January 2024 it should read UTC partitions
`(2023,12)` and `(2024,1)`, from `2023-12-31 14:00:00` to `2024-01-31 14:00:00`.

In [7]:
print(aest_month_window(2024, 1))   # -> ('2023-12-31 14:00:00', '2024-01-31 14:00:00', [(2023,12),(2024,1)])
print(aest_month_window(2024, 7))
print(aest_month_window(2025, 12))

('2023-12-31 14:00:00', '2024-01-31 14:00:00', [(2023, 12), (2024, 1)])
('2024-06-30 14:00:00', '2024-07-31 14:00:00', [(2024, 6), (2024, 7)])
('2025-11-30 14:00:00', '2025-12-31 14:00:00', [(2025, 11), (2025, 12)])


In [ ]:
print(vv.preview_sql(year=2024, month=1, n_parts=N_PARTS, part=0)[:4000])

In [13]:
preview = vv.preview_sql(
    year=2024,
    month=1,
    n_parts=N_PARTS,
    part=0,
)

# The inherited low/moderate-P capability branches must be gone.
for obsolete_fragment in (
    "0.2 * S_99",
    "0.6 * S_99",
    "0.8 * S_99",
):
    assert obsolete_fragment not in preview, (
        f"Old capability branch remains: {obsolete_fragment}"
    )

# The generated INSERT must contain the physical S-circle.
assert "power(S_99, 2) - power(abs(P_kW), 2)" in preview

print("Volt-VAr builder is using the pure S-circle capability.")
print(preview[:4000])

Volt-VAr builder is using the pure S-circle capability.

    INSERT INTO conformance_voltvar_v2
    WITH
    
    data AS (
        SELECT
            m.site_id,
            ts.t_stamp,
            sum(ts.power * m.circuit_polarity) / 1000 AS P_kW,
            sum(ts.energy_reactive * m.circuit_polarity) / 1000 * 12 AS Q_kvar,
            max(ts.voltage)        AS V,               -- original was avg(voltage)
            max(m.ac_capacity_kw)  AS ac_capacity_kw,  -- nameplate: drives the STANDARD's curves
            max(m.s_99)            AS S_99             -- empirical: drives the CAPABILITY curve 
        FROM ts
        JOIN (
            SELECT circuit_id,
                max(site_id)          AS site_id,
                max(circuit_polarity) AS circuit_polarity,
                max(ac_capacity_kw)   AS ac_capacity_kw,
                max(s_99)             AS s_99
            FROM meta_up23c
            WHERE is_pv = True AND ac_capacity_kw > 0 AND s_99 > 0 AND flex_export_detect

## 2. Create the empty tables

Destructive: drops and recreates. `_v2` suffix throughout.
(originals never touched)

In [14]:
print(vv.create_table(aq, database=DB))
print(vw.create_table_basic(aq, database=DB))
print(vw.create_table_ghi(aq, database=DB))
time.sleep(5)   
tables(DB)[tables(DB)["Table"].str.contains("conformance")][["Table"]]

Created empty conformance_voltvar_v2
Created empty conformance_voltwatt_v2
Created empty conformance_voltwattghi_v2


,Table
3,conformance_antiisland
4,conformance_sust_op
5,conformance_sust_op_3w
6,conformance_voltvar
7,conformance_voltvar_v2
8,conformance_voltwatt
9,conformance_voltwatt_v2
10,conformance_voltwattghi
11,conformance_voltwattghi_v2
15,review_conformance_sust_op


## 3. Test one AEST month, one site slice

`parts=[0]` is 1/8 of sites for AEST January 2024. This should complete in a few minutes. 

In [15]:
vv.run_months_voltvar(aq, database=DB, year=2024, months=[1],
                      n_parts=N_PARTS, parts=[0], exclude_flex=True)

loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])


['loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])']

In [16]:
vw.run_months_basic(aq, database=DB, year=2024, months=[1],
                    n_parts=N_PARTS, parts=[0], exclude_flex=True)

loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])


['loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])']

In [17]:
vw.run_months_ghi(aq, database=DB, year=2024, months=[1],
                  n_parts=N_PARTS, parts=[0], exclude_flex=True)

loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])


['loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])']

## 4. Validate the test

Every "MUST be 0" line must actually be 0 before you go any further.

The one to watch is **duplicate keys**. If that is non-zero, the AEST window logic has leaked and a site-day has been split across two INSERTs.

In [18]:
vv.validate(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1   83073     1418      31

Duplicate (year,month,day,day_night,site_id) keys (MUST be 0): 0

Coherence (all MUST be 0):
 neg_curtailment_rows  bucket_mismatch_rows  count_inversion_rows
                    0                     0                     0

Counterfactual coverage in the V-VAr curtailment zone:
 eligible  no_counterfactual  pct_missing
     4090               3730         91.2


In [19]:
vw.validate_basic(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1   83073     1418      31

Duplicate keys (MUST be 0): 0

Coherence (all MUST be 0):
 neg_rows  count_inversion_rows  impossible_rows
        0                     0                0


In [20]:
vw.validate_ghi(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1   83073     1418      31

Duplicate keys (MUST be 0): 0

Coherence (all MUST be 0):
 neg_curtailment_rows  impossible_rows  count_inversion_rows
                    0                0                     0

Counterfactual coverage above 253 V:
 exposed_intervals  no_counterfactual  pct_missing
             40439              29125        72.02


In [21]:
# Eyeball actual rows. 
# Check: day spans 1..31, day_night has both values,
# and P_kW_sum is positive during the day.
aq(f"""
    SELECT site_id, year, month, day, day_night,
           round(P_kW_sum, 1)                   AS P_kW_sum,
           round(nonconformance_voltvar_sum, 3) AS nonconf,
           round(curtailment_voltvar_sum, 3)    AS curtail,
           curtailment_eligible_count, null_uncurtailed_P_count,
           exposed_count, all_intervals_count, total_count
    FROM {vv.TARGET}
    ORDER BY curtailment_voltvar_sum DESC NULLS LAST
    LIMIT 15
""", database=DB)

,site_id,year,month,day,day_night,P_kW_sum,nonconf,curtail,curtailment_eligible_count,null_uncurtailed_P_count,exposed_count,all_intervals_count,total_count
0,616600992,2024,1,18,day,11900.8,307.330,8.862,16,0,144,144,144
1,616600992,2024,1,19,day,11765.9,295.046,7.826,6,3,144,144,144
2,46014256,2024,1,2,day,544.7,8.787,1.739,1,0,117,144,144
3,1147413752,2024,1,19,day,7417.8,5.672,1.324,2,1,52,144,144
4,476487840,2024,1,29,day,2870.2,21.682,1.205,11,2,144,144,144
5,375244288,2024,1,9,day,553.1,0.515,0.712,1,0,111,144,144
6,114611480,2024,1,18,day,1340.6,25.131,0.668,5,2,144,144,144
7,672712144,2024,1,24,day,439.5,158.943,0.625,4,0,49,144,144
8,1345392728,2024,1,12,day,451.3,1.236,0.538,3,0,144,144,144
9,234051848,2024,1,13,day,702.7,2.005,0.498,9,7,82,144,144


In [22]:
# regression test: the AEST boundary.
# Under original UTC extraction, intervals from 00:00-09:55 AEST were booked to
# the previous day. Here, day 1 of the month must contain a full AEST day --
# including its early-morning (night) intervals, which live in the PREVIOUS UTC
# month's partition. If day=1 has far fewer intervals than day=2, the window
# logic is wrong.
# NOTE: Since the data starts on 2024, we are missing the 10h from 31-DEC-2023, so day=1 will have 10 fewer intervals than day=2.
aq(f"""
    SELECT day, sum(all_intervals_count) AS intervals, count(DISTINCT site_id) AS sites
    FROM {vv.TARGET}
    WHERE year = 2024 AND month = 1 AND day IN (1, 2, 15, 30, 31)
    GROUP BY day ORDER BY day
""", database=DB)

,day,intervals,sites
0,1,216437,1330
1,2,380688,1333
2,15,382973,1346
3,30,390229,1368
4,31,390601,1368


## 5. Full load (2024 **and** 2025)

Each call is one AEST month * 8 site-slices = 8 Athena queries. 
A full year is 96 queries per table. 
Run one table at a time and check the printout.

If Athena throttles (`TooManyRequestsException`), 
drop `N_PARTS` to 4 or run `months` in two halves.

In [23]:
# Remove all test-slice output before the production run.
# These functions drop and recreate only the _v2 Stage 2 tables.
print(vv.create_table(aq, database=DB))
print(vw.create_table_basic(aq, database=DB))
print(vw.create_table_ghi(aq, database=DB))

print("Test results cleared; empty production tables recreated.")

Created empty conformance_voltvar_v2
Created empty conformance_voltwatt_v2
Created empty conformance_voltwattghi_v2
Test results cleared; empty production tables recreated.


In [24]:
'''
# Volt-VAr, 2024. Part 0 of Jan is already loaded from above
# rerun it and you WILL double-count. Load Jan parts 1..7:
vv.run_months_voltvar(aq, database=DB, year=2024, months=[1],
                      n_parts=N_PARTS, parts=list(range(1, N_PARTS)))
# then Feb-Dec fully:
vv.run_months_voltvar(aq, database=DB, year=2024, months=list(range(2, 13)),
                      n_parts=N_PARTS)

vv.run_months_voltvar(aq, database=DB, year=2025, months=list(range(1, 13)),
                      n_parts=N_PARTS)
'''

MONTHS = list(range(1, 13))

# Volt-VAr: complete 2024 and 2025.
for year in (2024, 2025):
    vv.run_months_voltvar(
        aq,
        database=DB,
        year=year,
        months=MONTHS,
        n_parts=N_PARTS,
        exclude_flex=True,
    )

loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=1/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=2/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=3/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=4/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=5/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=6/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=7/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-02 part=0/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions

In [34]:
# Basic Volt-Watt: complete 2024 and 2025.
for year in (2024, 2025):
    vw.run_months_basic(
        aq,
        database=DB,
        year=year,
        months=MONTHS,
        n_parts=N_PARTS,
        exclude_flex=True,
    )

loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=1/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=2/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=3/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=4/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=5/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=6/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=7/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-02 part=0/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions

In [35]:
# GHI-aware Volt-Watt: complete 2024 and 2025.
for year in (2024, 2025):
    vw.run_months_ghi(
        aq,
        database=DB,
        year=year,
        months=MONTHS,
        n_parts=N_PARTS,
        exclude_flex=True,
    )

loaded AEST 2024-01 part=0/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=1/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=2/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=3/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=4/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=5/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=6/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-01 part=7/8 (UTC 2023-12-31 14:00:00 -> 2024-01-31 14:00:00, partitions [(2023, 12), (2024, 1)])
loaded AEST 2024-02 part=0/8 (UTC 2024-01-31 14:00:00 -> 2024-02-29 14:00:00, partitions

## 6. Full validation

In [36]:
vv.validate(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1  667804    11339      31
 2024      2  641181    11540      29
 2024      3  692436    11522      31
 2024      4  675848    11631      30
 2024      5  698418    11733      31
 2024      6  668280    11675      30
 2024      7  662586    11303      31
 2024      8  657899    11068      31
 2024      9  635615    11019      30
 2024     10  657154    11072      31
 2024     11  640374    11158      30
 2024     12  670444    11165      31
 2025      1  671797    11357      31
 2025      2  598275    11123      28
 2025      3  633925    10769      31
 2025      4  614536    10552      30
 2025      5  628434    10542      31
 2025      6  517058    10269      30
 2025      7  618363    10458      31
 2025      8  606885    10147      31
 2025      9  573473     9975      30
 2025     10  579826     9766      31
 2025     11  544404     9463      30
 2025     12  546209     9183      31

Duplicate (ye

In [37]:
vw.validate_basic(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1  667804    11339      31
 2024      2  641181    11540      29
 2024      3  692436    11522      31
 2024      4  675848    11631      30
 2024      5  698418    11733      31
 2024      6  668280    11675      30
 2024      7  662586    11303      31
 2024      8  657899    11068      31
 2024      9  635615    11019      30
 2024     10  657154    11072      31
 2024     11  640374    11158      30
 2024     12  670444    11165      31
 2025      1  671797    11357      31
 2025      2  598275    11123      28
 2025      3  633925    10769      31
 2025      4  614536    10552      30
 2025      5  628434    10542      31
 2025      6  517058    10269      30
 2025      7  618363    10458      31
 2025      8  606885    10147      31
 2025      9  573473     9975      30
 2025     10  579826     9766      31
 2025     11  544404     9463      30
 2025     12  546209     9183      31

Duplicate key

In [38]:
vw.validate_ghi(aq, database=DB);

Rows / sites / days per AEST month:
 year  month  n_rows  n_sites  n_days
 2024      1  667804    11339      31
 2024      2  641181    11540      29
 2024      3  692436    11522      31
 2024      4  675848    11631      30
 2024      5  698418    11733      31
 2024      6  668280    11675      30
 2024      7  662586    11303      31
 2024      8  657899    11068      31
 2024      9  635615    11019      30
 2024     10  657154    11072      31
 2024     11  640374    11158      30
 2024     12  670444    11165      31
 2025      1  671797    11357      31
 2025      2  598275    11123      28
 2025      3  633925    10769      31
 2025      4  614536    10552      30
 2025      5  628434    10542      31
 2025      6  517058    10269      30
 2025      7  618363    10458      31
 2025      8  606885    10147      31
 2025      9  573473     9975      30
 2025     10  579826     9766      31
 2025     11  544404     9463      30
 2025     12  546209     9183      31

Duplicate key

In [39]:
# R13 regression test: the two Volt-Watt tables must share a denominator.
vw.cross_check(aq, database=DB);

Basic versus GHI Volt-Watt cross-check (all MUST be 0):
 keys_missing_from_basic  keys_missing_from_ghi  total_count_mismatches  all_intervals_mismatches  power_sum_mismatches
                       0                      0                       0                         0                     0


In [40]:
# If no duplicates in the source, compare the tables directly
aq(f"""
    SELECT b.total_count AS basic_tc, g.total_count AS ghi_tc,
           g.total_count - b.total_count AS diff,
           b.all_intervals_count AS basic_all, g.all_intervals_count AS ghi_all
    FROM {vw.TARGET_BASIC} b
    JOIN {vw.TARGET_GHI} g
      ON b.site_id = g.site_id AND b.year = g.year
     AND b.month = g.month AND b.day = g.day AND b.day_night = g.day_night
    WHERE b.total_count <> g.total_count
    LIMIT 10
""", database=DB)

#Empty table = success

,basic_tc,ghi_tc,diff,basic_all,ghi_all


## 7. Reconcile against original tables

Differences are **expected**. They should be fully explained by:

- flex-export sites excluded (biggest effect; ~700 sites in Stage 1)
- `max(voltage)` is >= `avg(voltage)`, so more intervals cross 240 V / 253 V
- AEST day boundaries reshuffle intervals between days
- capability clamped on `s_99`, not nameplate

If the site-count drop does **not** match the flex-export count, stop and find out why before trusting anything.

In [41]:
vv.compare_to_original(aq, database=DB, original="conformance_voltvar");

sites in conformance_voltvar_v2:  15,609
sites in conformance_voltvar: 16,147
sites dropped:       539
flex-export sites (expected explanation for the drop): 539


In [42]:
# Fleet-level headline comparison. Expect the same order of magnitude, not
# identical numbers.
aq(f"""
    SELECT 'v2' AS tbl, year,
           round(sum(nonconformance_voltvar_sum), 0)  AS nonconf_kvar,
           round(sum(curtailment_voltvar_sum), 0)     AS curtail_kw,
           sum(total_count)                           AS intervals
    FROM {vv.TARGET} GROUP BY year
    ORDER BY year
""", database=DB)

,tbl,year,nonconf_kvar,curtail_kw,intervals
0,v2,2024,890392691.0,1136.0,1136185798
1,v2,2025,760644356.0,486.0,1007088082


## 8. the number you cannot publish yet

`nonconformance_voltvar_red_sum` reproduces Hossein's definition:
`adverse + inactive + near_conformant` — i.e. it counts the sites that are
essentially complying and ignores the ones falling well short.

`nonconformance_voltvar_red_alt_sum` is `adverse + inactive + significant_shortfall`
— what the definition almost certainly should be.

Run this, take both numbers to Baran, and get him to confirm which range CANVAS
intended before either appears in the paper.

In [43]:
aq(f"""
    SELECT year,
           round(sum(nonconformance_voltvar_red_sum), 0)     AS red_sum,
           sum(nonconformance_voltvar_red_count)             AS red_count,
           round(sum(Q_adverse_sum), 0)                      AS adverse,
           round(sum(Q_inactive_sum), 0)                     AS inactive,
           round(sum(Q_near_conformant_sum), 0)              AS near_conformant,
           round(sum(Q_significant_shortfall_sum), 0)        AS significant_shortfall
    FROM {vv.TARGET}
    GROUP BY year ORDER BY year
""", database=DB)

,year,red_sum,red_count,adverse,inactive,near_conformant,significant_shortfall
0,2024,871944621.0,573929328,120547811.0,723633127.0,450975.0,27763684.0
1,2025,745785827.0,484104174,98716066.0,621377312.0,408481.0,25692449.0


In [44]:
aq("""
    SELECT count(*) AS n_duplicate_keys
    FROM (
        SELECT site_id, t_stamp
        FROM all_uncurtailedpv_v2
        GROUP BY site_id, t_stamp
        HAVING count(*) > 1
    )
""", database=DB)

,n_duplicate_keys
0,0
